# Lab | LangChain Evaluation

## Intro

In this lab we follow the LangSmith tutorial **[Evaluate a RAG application](https://docs.langchain.com/langsmith/evaluate-rag-tutorial)** (LangChain v1).

We will:
1. Build a small RAG bot (index → retrieve → generate)
2. Create a dataset of questions + reference answers in LangSmith
3. Define four LLM-as-judge evaluators: **correctness**, **relevance**, **groundedness**, **retrieval relevance**
4. Run an experiment and inspect the results
5. Re-run the same evaluation on a different dataset (the NYC text in `data/`)

The point is for you to understand all the steps involved and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## Setup

### 1. Create a virtual environment (recommended ⚠️)

Run this in a terminal from the project folder:

```bash
python3 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install -U pip
pip install -r requirements.txt
python -m ipykernel install --user --name lab-langchain-eval --display-name "Python (lab-langchain-eval)"
```

Then select the **Python (lab-langchain-eval)** kernel in Jupyter / VS Code.

### 2. API keys

Create a `.env` file in the project folder (it is git-ignored):

```
OPENAI_API_KEY=sk-...
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_TRACING=true
```

Get a LangSmith API key at https://smith.langchain.com (Settings → API Keys).

### Install dependencies
Only needed if you did not install `requirements.txt` above.

In [1]:
%pip install -U langsmith "langchain[openai]" langchain-text-splitters bs4 requests python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


### Configure the environment

In [2]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "YOUR LANGSMITH API KEY"   # or set it in .env
# os.environ["OPENAI_API_KEY"] = "YOUR OPENAI API KEY"         # or set it in .env

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing"
assert os.getenv("LANGSMITH_API_KEY"), "LANGSMITH_API_KEY is missing"

## Build the application

The app has three stages: **indexing**, **retrieval** and **generation**. We use LangChain here, but the evaluation patterns work with any framework.

### Index documents

Load Lilian Weng's blog posts, split them into chunks and index them in an in-memory vector store.

In [3]:
import bs4
import requests
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Minimal helper to load a web page as a LangChain Document
def load_web_page(url: str, bs_kwargs: dict | None = None) -> list[Document]:
    response = requests.get(url)
    response.raise_for_status()
    soup = bs4.BeautifulSoup(response.text, "html.parser", **(bs_kwargs or {}))
    return [Document(page_content=soup.get_text(), metadata={"source": url})]

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
docs_list = [
    doc
    for url in urls
    for doc in load_web_page(url, bs_kwargs={"parse_only": bs4_strainer})
]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Add the document chunks to the "vector store" using OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=OpenAIEmbeddings(),
)

# With langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(k=6)

print(f"{len(docs_list)} documents -> {len(doc_splits)} chunks")

3 documents -> 183 chunks


### Generate answers

The `@traceable` decorator sends each run of `rag_bot` to LangSmith, so you can inspect retrieval and generation step by step.

In [4]:
from langchain_openai import ChatOpenAI
from langsmith import traceable

llm = ChatOpenAI(model="gpt-5.5", temperature=1)

# Add decorator so this function is traced in LangSmith
@traceable()
def rag_bot(question: str) -> dict:
    # LangChain retriever will be automatically traced
    docs = retriever.invoke(question)
    docs_string = "".join(doc.page_content for doc in docs)
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.
       Use the following source documents to answer the user's questions.
       If you don't know the answer, just say that you don't know.
       Use three sentences maximum and keep the answer concise.

<context>
{docs_string}
</context>"""
    # langchain ChatModel will be automatically traced
    ai_msg = llm.invoke([
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )
    return {"answer": ai_msg.content, "documents": docs}

In [5]:
# Quick test
result = rag_bot("What are the types of biases that can arise with few-shot prompting?")
print(result["answer"])

The biases that can arise with few-shot prompting are:

1. **Majority label bias**: the model favors labels that appear more often in the examples.
2. **Recency bias**: the model tends to repeat the label that appears near the end of the prompt.
3. **Common token bias**: the model is more likely to output common tokens than rare ones.


## Create a dataset

A small dataset of questions and reference answers, stored in LangSmith.

In [6]:
from langsmith import Client

client = Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    },
]

# Create the dataset and examples in LangSmith (skip if it already exists, so the cell can be re-run)
dataset_name = "Lilian Weng Blogs Q&A"
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(dataset_name=dataset_name)
    client.create_examples(dataset_id=dataset.id, examples=examples)
else:
    print(f"Dataset '{dataset_name}' already exists")

## Define evaluators

| Evaluator | Compares | Needs reference answer? |
|---|---|---|
| **Correctness** | response vs. reference answer | ✅ |
| **Relevance** | response vs. input question | ❌ |
| **Groundedness** | response vs. retrieved docs | ❌ |
| **Retrieval relevance** | retrieved docs vs. input question | ❌ |

Each one is an LLM-as-judge with structured output (an `explanation` followed by a boolean score).

### Correctness: response vs. reference answer

In [8]:
from typing_extensions import Annotated, TypedDict

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

# Grade prompt
correctness_instructions = """You are a teacher grading a quiz. You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. (2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
grader_llm = ChatOpenAI(model="gpt-5.5", temperature=0).with_structured_output(
    CorrectnessGrade, method="json_schema", strict=True
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""
    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

### Relevance: response vs. input

In [9]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[
        bool, ..., "Provide the score on whether the answer addresses the question"
    ]

# Grade prompt
relevance_instructions = """You are a teacher grading a quiz. You will be given a QUESTION and a STUDENT ANSWER. Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatOpenAI(model="gpt-5.5", temperature=0).with_structured_output(
    RelevanceGrade, method="json_schema", strict=True
)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Groundedness: response vs. retrieved docs

In [10]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[
        bool, ..., "Provide the score on if the answer hallucinates from the documents"
    ]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. You will be given FACTS and a STUDENT ANSWER. Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. (2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
grounded_llm = ChatOpenAI(model="gpt-5.5", temperature=0).with_structured_output(
    GroundedGrade, method="json_schema", strict=True
)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["grounded"]

### Retrieval relevance: retrieved docs vs. input

In [11]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[
        bool,
        ...,
        "True if the retrieved documents are relevant to the question, False otherwise",
    ]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. You will be given a QUESTION and a set of FACTS provided by the student. Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatOpenAI(
    model="gpt-5.5", temperature=0
).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"
    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

## Run the evaluation

Run the app against every example in the dataset and score it with all four evaluators. The results show up as an **experiment** in LangSmith (follow the link printed below).

In [12]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-5.5"},
)

View the evaluation results for experiment: 'rag-doc-relevance-9480d010' at:
https://smith.langchain.com/o/f2795715-5d87-47c5-96f4-391ef13fe4e1/datasets/e25977c3-dc74-499c-bc18-901bb92c8bd7/compare?selectedSessions=8d732464-1ded-416a-bf9a-e213a4cd7841




In [13]:
# Explore results locally as a dataframe
df = experiment_results.to_pandas()
df

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What are the types of biases that can arise wi...,The biases that can arise with few-shot prompt...,[page_content='Zero-shot and few-shot learning...,None,The biases that can arise with few-shot prompt...,True,True,True,True,4.054033,dc4a35f7-4806-4ada-a5a2-f9963dea9f91,01a1a28a-5775-7911-8b55-213f5bc2e7a0
1,How does the ReAct agent use self-reflection?,ReAct does not explicitly use self-reflection ...,[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",True,True,True,True,4.072136,fc6c4843-0261-482e-9cab-4575c304691b,01a1a28a-9b4b-7377-9c6c-7a435b1a2f10
2,What are five types of adversarial attacks?,Five types of adversarial attacks are:\n\n1. T...,[page_content='Black-box attacks assume that a...,None,Five types of adversarial attacks are (1) Toke...,True,True,True,True,2.393427,fdef5cee-631e-412a-b6ca-210bec22523e,01a1a28a-f763-7555-87f4-a646529e2187


## Your turn: evaluate on a different dataset

Re-use the exact same pipeline and evaluators on `data/nyc_text.txt`. Only the **index** and the **dataset** change — the evaluators stay the same. Compare the scores in LangSmith with the Lilian Weng experiment.

Try also changing `chunk_size`, `k`, the model, or the prompt and re-running to see how the scores move.

In [14]:
# Load the NYC text and build a new retriever
with open("data/nyc_text.txt", encoding="utf-8") as f:
    nyc_docs = [Document(page_content=f.read(), metadata={"source": "nyc_text.txt"})]

nyc_splits = text_splitter.split_documents(nyc_docs)
nyc_vectorstore = InMemoryVectorStore.from_documents(nyc_splits, embedding=OpenAIEmbeddings())

# rag_bot uses the global `retriever`, so we point it at the new vector store
retriever = nyc_vectorstore.as_retriever(k=6)
print(f"{len(nyc_splits)} chunks")

179 chunks


In [15]:
nyc_examples = [
    {"inputs": {"question": "What is the population of New York City as of 2020?"},
     "outputs": {"answer": "8,804,190"}},
    {"inputs": {"question": "Which borough of New York City has the highest population?"},
     "outputs": {"answer": "Brooklyn"}},
    {"inputs": {"question": "What is the economic significance of New York City?"},
     "outputs": {"answer": "New York City is a global financial capital, home to Wall Street and major financial institutions, with a diverse economy spanning technology, media, healthcare, education, real estate and tourism."}},
    {"inputs": {"question": "How did New York City get its name?"},
     "outputs": {"answer": "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor."}},
    {"inputs": {"question": "What is the significance of the Statue of Liberty in New York City?"},
     "outputs": {"answer": "The Statue of Liberty is a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants arriving by ship in the late 19th and early 20th centuries."}},
]

nyc_dataset_name = "NYC Q&A"
if not client.has_dataset(dataset_name=nyc_dataset_name):
    nyc_dataset = client.create_dataset(dataset_name=nyc_dataset_name)
    client.create_examples(dataset_id=nyc_dataset.id, examples=nyc_examples)
else:
    print(f"Dataset '{nyc_dataset_name}' already exists")

In [16]:
nyc_results = client.evaluate(
    target,
    data=nyc_dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-nyc",
    metadata={"version": "nyc_text, gpt-5.5"},
)
nyc_results.to_pandas()

View the evaluation results for experiment: 'rag-nyc-e494a89f' at:
https://smith.langchain.com/o/f2795715-5d87-47c5-96f4-391ef13fe4e1/datasets/f36e469f-6033-4179-b9d7-409bead74c27/compare?selectedSessions=2695d8f0-fbd1-4435-afbe-8f1ed04b3636




,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,How did New York City get its name?,New York City was named in 1664 after the Duke...,"[page_content='== Etymology ==\n\nIn 1664, New...",None,New York City got its name when it came under ...,True,True,True,True,2.826232,17835d9c-db7d-4d08-aa64-a1642b4b2167,01a1a28b-aa3e-79ba-8fd3-cfafdcf51788
1,What is the population of New York City as of ...,"New York City’s population as of 2020 was **8,...",[page_content='New York City is the most popul...,None,"8,804,190",True,True,True,True,1.922062,1b00a9db-141f-4ed3-9674-bd412d49dd7f,01a1a28b-f7c8-7153-bf4e-28ec0fc608aa
2,What is the significance of the Statue of Libe...,The Statue of Liberty is a symbol of the Unite...,[page_content='The Statue of Liberty National ...,None,The Statue of Liberty is a symbol of the Unite...,True,True,True,True,2.702077,213ae1a4-2914-4ef1-ad88-fcbae19d749b,01a1a28c-285a-705f-a44c-888851b7824b
3,Which borough of New York City has the highest...,Brooklyn has the highest population of New Yor...,[page_content='Manhattan (New York County) is ...,None,Brooklyn,True,True,True,True,2.466376,32f78b83-a662-4ebe-a3a4-7ae67eee5d0b,01a1a28c-6e80-7627-afa9-7b52477b87e7
4,What is the economic significance of New York ...,New York City is a major global economic cente...,[page_content='New York City's most important ...,None,"New York City is a global financial capital, h...",True,True,True,True,3.368385,c3aac51c-5227-4697-97dd-893f0a57b2f5,01a1a28c-a8e6-78f3-afe3-6cfaaf60ff6e


## Reflection

- Which evaluator was the strictest? Which one failed most often, and why?
- Look at a failing example in LangSmith: was the problem **retrieval** (wrong docs) or **generation** (bad answer from good docs)?
- Which evaluators need a reference answer, and which could you run on live production traffic?
- How did changing `chunk_size` / `k` affect groundedness and retrieval relevance?